# StravaGANte Syntetic Data Retriever
The dataset is collected using OpenRouteService, an open source project which exposes free APIs.

In [6]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
import serpapi
import csv
import sys
from time import sleep

IMDB 250 top films to scrape locations

In [ ]:
def print_progress_bar(iteration, total, length=50):
    percent = ("{0:.1f}").format(100 * (iteration / float(total)))
    filled_length = int(length * iteration // total)
    bar = '█' * filled_length + '-' * (length - filled_length)
    sys.stdout.write(f'\rProgress: |{bar}| {percent}%\n')
    sys.stdout.flush()

latlong_file = '../Data/latlong_movies.csv'
df_movies = pd.read_csv('../Data/IMDB_Top_250_Movies.csv', usecols=[1])
movie_names = df_movies['name'].tolist()
total_rows = len(movie_names)

movie_locations = {}

if os.path.isfile(latlong_file):
    existing_data = pd.read_csv(latlong_file)
    existing_data.dropna(subset=['name', 'latlong_url'], inplace=True)
    existing_data.to_csv(latlong_file, index=False)
    existing_movies = existing_data['name'].tolist()
    completed_rows = existing_data['latlong_url'].notna().sum()
else:
    existing_movies = []
    with open(latlong_file, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['name', 'latlong_url'])

load_dotenv()
# serpapi_api_key = os.getenv("serpapi_token")
serpapi_api_key = os.getenv("serpapi_token_secondary")
print("Token ok.") if serpapi_api_key else print("Token not found.")

for movie in movie_names:
    if movie not in existing_movies:
        print(f"{movie}...")
        query = f'site:latlong.net/location/ {movie}'
        try:
            search = serpapi.search({
                "q": query,
                "location": "United States",
                "api_key": serpapi_api_key
            })
            results = search.get_dict()
            href = results['organic_results'][0]['link'] if 'organic_results' in results and results['organic_results'] else None
        except Exception as e:
            print(f"Error occurred: {e}")
            href = None
        
        if href:
            with open(latlong_file, mode='a', newline='') as file:
                writer = csv.writer(file)
                writer.writerow([movie, href])
            print(f"Found.")
            completed_rows += 1
        else:
            print(f"No results found.")
        
    print_progress_bar(completed_rows, total_rows)


Scrape from latlong.net every location.

In [30]:
import requests
from bs4 import BeautifulSoup

latlong_movies_copy_file = '../Data/latlong_movies.csv'
output_file = '../Data/latlong_movies_coordinates.csv'

# Read the CSV file
df_latlong = pd.read_csv(latlong_movies_copy_file)

# Create a list to store the results
results = []

# Iterate over each row in the DataFrame
for index, row in df_latlong.iterrows():
    url = row['latlong_url']
    movie_name = row['name']
    
    # Make a request to the URL
    response = requests.get(url)
    
    if response.status_code == 200:
        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the table in the page
        table = soup.find('table')
        
        if table:
            rows = table.find_all('tr')
            for r in rows[1:]:
                lat = r.find_all('td')[1].text.strip()
                lon = r.find_all('td')[2].text.strip()
                location_name = r.find_all('td')[0].text.strip()
            
                # Append the result to the list
                results.append([movie_name, location_name, lat, lon])
        else:
            print(f"No table found for {movie_name}")
    else:
        print(f"Failed to retrieve {url}")

# Create a DataFrame from the results
df_results = pd.DataFrame(results, columns=['name', 'location_name', 'latitude', 'longitude'])

# Save the results to a new CSV file
df_results.to_csv(output_file, index=False)

POST Request to OpenRouteService

In [21]:
import random
from geopy.distance import geodesic
from datetime import datetime, timedelta

MAX_ATTEMPTS = 5

# Access Limits: (daily / per minute)
# Directions* (2.000 / 40) 
# --> 1 REQUEST EVERY 43.2 SECONDS!

profiles = [
    'driving-car',
    'driving-hgv',
    'cycling-regular',
    'cycling-road',
    'cycling-mountain',
    'cycling-electric'
]
output_filedir = '../Data/Syntetic/'
os.makedirs(output_filedir, exist_ok=True)
locs_file = '../Data/latlong_movies_coordinates.csv'
n_call = 0
n_call_fails = 0

load_dotenv()
ors_token = os.getenv("OpenRouteServiceApiKey")
print("Token ok.") if ors_token else print("Token not found.")

def generate_waypoint(pz, distance_km):
    bearing = random.uniform(0, 360)
    destination = geodesic(kilometers=distance_km).destination((pz[1], pz[0]), bearing)
    return [destination.longitude, destination.latitude]

def calculate_wait_time(requests_made_today=0, requests_made_this_minute=0):
    now = datetime.now()
    seconds_in_day = 86400
    seconds_in_minute = 60

    # Calculate remaining requests for the day and minute
    remaining_requests_today = 2000 - requests_made_today
    remaining_requests_this_minute = 40 - requests_made_this_minute

    # Calculate the time left in the day and minute
    time_left_in_day = (datetime.combine(now.date() + timedelta(days=1), datetime.min.time()) - now).total_seconds()
    time_left_in_minute = seconds_in_minute - now.second

    # Calculate wait times
    wait_time_day = time_left_in_day / remaining_requests_today if remaining_requests_today > 0 else seconds_in_day
    wait_time_minute = time_left_in_minute / remaining_requests_this_minute if remaining_requests_this_minute > 0 else seconds_in_minute

    # Return the maximum wait time needed
    return max(wait_time_day, wait_time_minute)

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': ors_token,
    'Content-Type': 'application/json; charset=utf-8'
}

df_locs = pd.read_csv(locs_file)

wait_time = calculate_wait_time()
print(f"Wait time: {wait_time} seconds.")

try:
    for index, row in df_locs[840:920].iterrows():
        gpx_folder = f'{output_filedir}{str(index).zfill(3)}/'
        os.makedirs(gpx_folder, exist_ok=True)
        
        for c in range(0, 5):
            gpx_file_path = os.path.join(gpx_folder, f'route_loc{str(index).zfill(3)}_{str(c)}.gpx')
            pz = [row['longitude'], row['latitude']]

            attempts = 0
            while not os.path.isfile(gpx_file_path):
                dist1 = random.uniform(5, 50)
                dist2 = random.uniform(5, 50)

                wp1 = generate_waypoint(pz, dist1)
                wp2 = generate_waypoint(pz, dist2)
                body = {"coordinates": [pz, wp1, wp2, pz]}
                call = requests.post(f'https://api.openrouteservice.org/v2/directions/{profiles[2]}/gpx', json=body, headers=headers)
                n_call+=1
                attempts+=1

                if call.status_code == 200:
                    with open(gpx_file_path, 'w') as file:
                        file.write(call.text)
                else:
                    print(f"Loc. {index} failed (gpx file n°{c}, attempt {attempts}). Error code: {call.status_code}") #\n   --> {call.text}")
                    n_call_fails += 1
                sleep(wait_time)

                if attempts > MAX_ATTEMPTS:
                    break

except KeyboardInterrupt:
    print("Execution interrupted by the user.")
finally:
    print(f"Stats: {n_call-n_call_fails}/{n_call} successful requests ({((n_call-n_call_fails)*100)/n_call:.1f}%).")

Token ok.
Wait time: 22.5633010125 seconds.
Loc. 842 failed (gpx file n°0, attempt 1). Error code: 404
Loc. 842 failed (gpx file n°0, attempt 2). Error code: 404
Loc. 842 failed (gpx file n°1, attempt 1). Error code: 404
Loc. 842 failed (gpx file n°2, attempt 1). Error code: 404
Loc. 842 failed (gpx file n°4, attempt 1). Error code: 404
Loc. 843 failed (gpx file n°2, attempt 1). Error code: 404
Loc. 843 failed (gpx file n°2, attempt 2). Error code: 404
Loc. 843 failed (gpx file n°2, attempt 3). Error code: 404
Loc. 843 failed (gpx file n°2, attempt 4). Error code: 404
Loc. 843 failed (gpx file n°2, attempt 5). Error code: 404
Loc. 843 failed (gpx file n°3, attempt 1). Error code: 404
Loc. 843 failed (gpx file n°3, attempt 2). Error code: 404
Loc. 843 failed (gpx file n°3, attempt 3). Error code: 404
Loc. 844 failed (gpx file n°1, attempt 1). Error code: 404
Loc. 844 failed (gpx file n°3, attempt 1). Error code: 404
Loc. 844 failed (gpx file n°3, attempt 2). Error code: 404
Loc. 844 fai

In [7]:
import os
from collections import defaultdict

directory = '../Data/Syntetic'
summary = defaultdict(int)

for root, dirs, files in os.walk(directory):
    if root == directory:
        continue  # Skip the root directory itself
    gpx_count = sum(1 for file in files if file.endswith('.gpx'))
    summary[gpx_count] += 1

total_folders = sum(summary.values())
print(f"Summary of .gpx files in each folder under Data/Syntetic ({total_folders} locations):")
for count in range(6):
    print(f" --> Loc with {count} .gpx files: {summary[count]} ( {summary[count] / total_folders * 100:.1f}% )")

# Check for missing folders
expected_folders = set(range(total_folders))
existing_folders = set(int(os.path.basename(root)) for root, dirs, files in os.walk(directory) if root != directory)
missing_folders = expected_folders - existing_folders

if missing_folders:
    print(f"Missing folders: {sorted(missing_folders)}")
else:
    print("\nNo folders are missing.")

locs_file = '../Data/latlong_movies_coordinates.csv'
df_locs = pd.read_csv(locs_file)
total_available_locations = len(df_locs)
print(f"\nLocations processed: {total_folders}/{total_available_locations} ({(total_folders / total_available_locations) * 100:.1f}%)")

Summary of .gpx files in each folder under Data/Syntetic (920 locations):
 --> Loc with 0 .gpx files: 59 ( 6.4% )
 --> Loc with 1 .gpx files: 17 ( 1.8% )
 --> Loc with 2 .gpx files: 21 ( 2.3% )
 --> Loc with 3 .gpx files: 44 ( 4.8% )
 --> Loc with 4 .gpx files: 77 ( 8.4% )
 --> Loc with 5 .gpx files: 702 ( 76.3% )

No folders are missing.

Locations processed: 920/2901 (31.7%)
